# Explorative Datenanalyse (EDA)

## Was ist eine EDA?

Bevor wir Modelle trainieren, schauen wir uns die Daten genau an. Das nennt man **Explorative Datenanalyse** (EDA). Dabei beantworten wir Fragen wie:

- Wie sehen unsere Daten aus? Gibt es Luecken oder Fehler?
- Welche Muster gibt es? (Tag/Nacht, Sommer/Winter, Wochentag/Wochenende)
- Welche Wettervariablen haengen mit der Residuallast zusammen?
- Wie stark haengt die Residuallast von ihren eigenen vergangenen Werten ab?

**Erinnerung:** Residuallast = Gesamtlast - Wind - Solar. Das ist die Leistung, die konventionelle Kraftwerke (Gas, Kohle, etc.) noch liefern muessen.

---

**Inhalt:**
1. Daten laden
2. Ueberblick und deskriptive Statistiken
3. Zeitreihenvisualisierung
4. Saisonale und tageszeitliche Muster
5. Verteilungen
6. Korrelationsanalyse (Strom und Wetter)
7. Vergleich der drei Wetter-Aggregationen
8. Stationaritaetstest
9. Autokorrelation
10. Zusammenfassung

In [ ]:
# Setup
import sys, os
sys.path.insert(0, os.path.abspath(".."))
os.chdir(os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import yaml
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.figsize": (14, 5), "figure.dpi": 150,
    "font.size": 11, "font.family": "serif",
})
sns.set_palette("colorblind")

with open("config/config.yaml") as f:
    config = yaml.safe_load(f)

print("Setup fertig.")

## 1. Daten laden

Wir haben zwei Datenquellen:
- **SMARD** (Bundesnetzagentur): Stundengenaue Stromdaten fuer ganz Deutschland
- **Open-Meteo**: Wetterdaten von 7 Standorten in Deutschland, gewichtet nach drei Schemata

In [ ]:
from pathlib import Path

smard = pd.read_csv("data/raw/smard_data.csv", index_col=0, parse_dates=True)
weather = pd.read_csv("data/raw/weather_data.csv", index_col=0, parse_dates=True)

print(f"=== SMARD Stromdaten ===")
print(f"  Zeilen: {smard.shape[0]:,} (stuendlich)")
print(f"  Spalten: {list(smard.columns)}")
print(f"  Zeitraum: {smard.index.min().date()} bis {smard.index.max().date()}")
print(f"  Fehlende Werte: {smard.isna().sum().sum()}")

print(f"\n=== Wetterdaten ===")
print(f"  Zeilen: {weather.shape[0]:,}")
print(f"  Spalten: {weather.shape[1]} (3 Gewichtungen x 6 Variablen = 18)")
print(f"  Zeitraum: {weather.index.min().date()} bis {weather.index.max().date()}")
print(f"  Fehlende Werte: {weather.isna().sum().sum()}")

## 2. Deskriptive Statistiken

Zuerst ein Ueberblick ueber die Zahlenwerte. Die wichtigsten Spalten:
- **residual_load**: Unser Zielwert (was wir vorhersagen wollen)
- **total_load**: Gesamter Stromverbrauch in Deutschland
- **solar**: Stromerzeugung durch Photovoltaik
- **wind_onshore/offshore**: Windkraft an Land / auf See
- **wind_total**: Wind gesamt (onshore + offshore)

In [ ]:
print("=== SMARD Stromdaten (alle Werte in Megawatt, MW) ===")
desc = smard.describe().round(0)
desc

**Was sagen uns diese Zahlen?**

- Die **Residuallast** schwankt zwischen ca. -9.500 MW (negativ! Erneuerbare erzeugen mehr als verbraucht wird) und +61.000 MW
- Der **Mittelwert** liegt bei ca. 33.000 MW — das ist die "typische" Residuallast
- **Solar** hat ein Minimum von 0 (nachts) und ein Maximum von ca. 40.000 MW
- **Wind** schwankt stark zwischen fast 0 und ueber 50.000 MW

## 3. Zeitreihenvisualisierung

Ein Blick auf den gesamten Zeitraum 2021-2025. Drei Grafiken:
1. Gesamtlast und Residuallast im Vergleich
2. Erneuerbare Erzeugung (Solar + Wind)
3. Residuallast als Monatsmittel (glaetter)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# 1) Gesamtlast vs Residuallast
axes[0].plot(smard.index, smard["total_load"], linewidth=0.3, label="Gesamtlast", color="gray")
axes[0].plot(smard.index, smard["residual_load"], linewidth=0.3, label="Residuallast", color="tab:blue")
axes[0].axhline(y=0, color="red", linestyle="--", linewidth=0.8, alpha=0.5)
axes[0].set_ylabel("MW")
axes[0].set_title("Gesamtlast und Residuallast (2021-2025)")
axes[0].legend()

# 2) Erneuerbare
axes[1].plot(smard.index, smard["solar"], linewidth=0.3, label="Solar", color="orange")
axes[1].plot(smard.index, smard["wind_total"], linewidth=0.3, label="Wind (Gesamt)", color="steelblue")
axes[1].set_ylabel("MW")
axes[1].set_title("Erneuerbare Erzeugung")
axes[1].legend()

# 3) Residuallast Monatsmittel
monthly = smard["residual_load"].resample("MS").agg(["mean", "std"])
axes[2].fill_between(monthly.index, monthly["mean"] - monthly["std"],
                      monthly["mean"] + monthly["std"], alpha=0.3, color="tab:blue")
axes[2].plot(monthly.index, monthly["mean"], linewidth=2, color="tab:blue")
axes[2].set_ylabel("MW")
axes[2].set_title("Residuallast — Monatsmittel +/- Standardabweichung")

plt.tight_layout()
plt.savefig("output/eda/01_zeitreihen_uebersicht.png", bbox_inches="tight")
plt.show()

**Was sieht man?**

- Die Residuallast (blau) liegt immer unter der Gesamtlast (grau) — logisch, weil Wind und Solar etwas abdecken
- Im **Sommer** ist die Residuallast viel niedriger (weniger Heizung + viel Solar)
- Im **Winter** ist sie hoch (viel Heizung + wenig Solar)
- Die Schwankungen sind enorm — von fast -10.000 MW bis +60.000 MW

## 4. Saisonale und tageszeitliche Muster

Die Residuallast folgt klaren Mustern:
- **Tageszeit**: Morgens steigt der Verbrauch (Industrie, Bueros), mittags Solar-Peak, abends Spitze
- **Jahreszeit**: Winter hoch, Sommer niedrig
- **Wochentag**: Werktags hoeher als am Wochenende

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Tagesprofil nach Saison
smard_copy = smard.copy()
smard_copy["hour"] = smard_copy.index.hour
smard_copy["month"] = smard_copy.index.month

season_map = {12: "Winter", 1: "Winter", 2: "Winter",
              3: "Fruehling", 4: "Fruehling", 5: "Fruehling",
              6: "Sommer", 7: "Sommer", 8: "Sommer",
              9: "Herbst", 10: "Herbst", 11: "Herbst"}
smard_copy["season"] = smard_copy["month"].map(season_map)

colors = {"Winter": "tab:blue", "Fruehling": "tab:green", "Sommer": "tab:orange", "Herbst": "tab:red"}
for season in ["Winter", "Fruehling", "Sommer", "Herbst"]:
    hourly = smard_copy.loc[smard_copy["season"] == season].groupby("hour")["residual_load"].mean()
    axes[0].plot(hourly.index, hourly.values, label=season, linewidth=2.5, color=colors[season])

axes[0].set_xlabel("Stunde des Tages")
axes[0].set_ylabel("Residuallast [MW]")
axes[0].set_title("Tagesprofil nach Jahreszeit")
axes[0].legend()
axes[0].set_xticks(range(0, 24, 2))
axes[0].grid(True, alpha=0.3)

# Wochenprofil
dow_names = ["Mo", "Di", "Mi", "Do", "Fr", "Sa", "So"]
weekly = smard_copy.groupby(smard_copy.index.dayofweek)["residual_load"].agg(["mean", "std"])
bars = axes[1].bar(range(7), weekly["mean"], yerr=weekly["std"], capsize=4,
                    edgecolor="black", linewidth=0.5, color="steelblue")
# Wochenende hervorheben
bars[5].set_color("lightcoral")
bars[6].set_color("lightcoral")
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(dow_names)
axes[1].set_ylabel("Residuallast [MW]")
axes[1].set_title("Residuallast nach Wochentag (rot = Wochenende)")
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("output/eda/02_saisonale_muster.png", bbox_inches="tight")
plt.show()

**Erkenntnisse:**

- **Winter**: Hohe Residuallast den ganzen Tag ueber. Morgens um 8 Uhr Spitze, mittags leichter Rueckgang (Solar hilft etwas)
- **Sommer**: Deutlich niedriger, besonders mittags — hier liefert Solar am meisten und drueckt die Residuallast nach unten
- **Wochenende** (rot): Ca. 5.000-8.000 MW weniger als werktags — Industrie und Bueros sind geschlossen

## 5. Verteilungen

Wie sind die Werte verteilt? Histogramme zeigen, welche Werte haeufig und welche selten vorkommen.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

plot_info = [
    ("residual_load", "Residuallast", "tab:blue"),
    ("total_load", "Gesamtlast", "gray"),
    ("solar", "Solar", "orange"),
    ("wind_onshore", "Wind Onshore", "steelblue"),
    ("wind_offshore", "Wind Offshore", "navy"),
    ("wind_total", "Wind Gesamt", "teal"),
]

for ax, (col, label, color) in zip(axes.flat, plot_info):
    data = smard[col].dropna()
    ax.hist(data, bins=80, edgecolor="black", linewidth=0.3, color=color, alpha=0.8)
    ax.axvline(data.mean(), color="red", linestyle="--", linewidth=1.5, label=f"Mittel: {data.mean():,.0f} MW")
    ax.set_title(label)
    ax.set_xlabel("MW")
    ax.legend(fontsize=8)

plt.suptitle("Verteilungen der Stromdaten", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("output/eda/03_verteilungen.png", bbox_inches="tight")
plt.show()

# Negative Residuallast
n_negative = (smard["residual_load"] < 0).sum()
pct_negative = n_negative / len(smard) * 100
print(f"\nNegative Residuallast: {n_negative} Stunden ({pct_negative:.1f}%)")
print(f"Das passiert, wenn Wind+Solar mehr erzeugen als Deutschland verbraucht.")

## 6. Korrelationsanalyse

**Korrelation** misst, wie stark zwei Variablen zusammenhaengen (von -1 bis +1):
- **+1**: Steigt A, steigt auch B (perfekter positiver Zusammenhang)
- **-1**: Steigt A, sinkt B (perfekter negativer Zusammenhang)
- **0**: Kein Zusammenhang

### 6.1 Korrelation der Energiedaten untereinander

In [ ]:
from src.data.preprocessing import combine_data
combined = combine_data(smard, weather, "1h")

# Energie-Korrelation
energy_cols = ["residual_load", "total_load", "solar", "wind_onshore", "wind_offshore", "wind_total"]
fig, ax = plt.subplots(figsize=(8, 6))
corr = combined[energy_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            mask=mask, square=True, ax=ax, vmin=-1, vmax=1)
ax.set_title("Korrelation der Energiedaten")
plt.tight_layout()
plt.savefig("output/eda/04_korrelation_energie.png", bbox_inches="tight")
plt.show()

**Was sieht man?**
- **Wind** korreliert stark negativ mit der Residuallast (ca. -0.8): Mehr Wind = weniger Residuallast
- **Solar** korreliert ebenfalls negativ (ca. -0.4): Mehr Sonne = weniger Residuallast
- **Gesamtlast** korreliert positiv (ca. +0.6): Mehr Verbrauch = mehr Residuallast

### 6.2 Korrelation mit Wetterdaten

Wir haben drei verschiedene Wetter-Gewichtungen:
- **weather_load_***: Gewichtet nach Bevoelkerung/Stromverbrauch (NRW schwer)
- **weather_wind_***: Gewichtet nach Windkapazitaet (Norddeutschland schwer)
- **weather_solar_***: Gewichtet nach Solarkapazitaet (Bayern/BaWue schwer)

In [ ]:
# Korrelation: Residuallast vs. alle 18 Wettervariablen
weather_cols = sorted([c for c in combined.columns if c.startswith("weather_")])
corr_with_residual = combined[weather_cols].corrwith(combined["residual_load"]).sort_values()

fig, ax = plt.subplots(figsize=(10, 8))
colors = []
for col in corr_with_residual.index:
    if "_load_" in col:
        colors.append("steelblue")
    elif "_wind_" in col:
        colors.append("teal")
    else:
        colors.append("orange")

ax.barh(range(len(corr_with_residual)), corr_with_residual.values, color=colors)
ax.set_yticks(range(len(corr_with_residual)))
# Kuerzere Labels
short_labels = [c.replace("weather_", "") for c in corr_with_residual.index]
ax.set_yticklabels(short_labels, fontsize=9)
ax.set_xlabel("Korrelation mit Residuallast")
ax.set_title("Wettervariablen vs. Residuallast\n(blau=Last-gew., gruen=Wind-gew., orange=Solar-gew.)")
ax.axvline(x=0, color="black", linewidth=0.8)
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.savefig("output/eda/05_korrelation_wetter.png", bbox_inches="tight")
plt.show()

print("\nTop-5 staerkste Korrelationen (absolut):")
top5 = corr_with_residual.abs().sort_values(ascending=False).head(5)
for col in top5.index:
    print(f"  {col}: {corr_with_residual[col]:+.3f}")

### 6.3 Scatter-Plots: Wetter vs. Residuallast

Hier sehen wir den Zusammenhang als Punktwolke. Jeder Punkt ist eine Stunde.
Wir verwenden jeweils die **am besten passende Gewichtung** pro Variable:
- Windgeschwindigkeit: **Wind-gewichtet** (dort wo die Windraeder stehen)
- Globalstrahlung: **Solar-gewichtet** (dort wo die Solaranlagen stehen)
- Temperatur: **Last-gewichtet** (dort wo die Menschen leben und Strom verbrauchen)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

scatter_vars = [
    ("weather_load_temperature_2m", "Temperatur [C]\n(Last-gewichtet)"),
    ("weather_wind_wind_speed_10m", "Wind 10m [m/s]\n(Wind-gewichtet)"),
    ("weather_wind_wind_speed_100m", "Wind 100m [m/s]\n(Wind-gewichtet)"),
    ("weather_solar_shortwave_radiation", "Globalstrahlung [W/m2]\n(Solar-gewichtet)"),
    ("weather_load_cloud_cover", "Bewoelkung [%]\n(Last-gewichtet)"),
    ("weather_load_relative_humidity_2m", "Luftfeuchtigkeit [%]\n(Last-gewichtet)"),
]

sample = combined.dropna().sample(min(5000, len(combined)), random_state=42)

for ax, (col, label) in zip(axes.flat, scatter_vars):
    if col in sample.columns:
        ax.scatter(sample[col], sample["residual_load"], alpha=0.1, s=3)
        ax.set_xlabel(label)
        ax.set_ylabel("Residuallast [MW]")
        
        # Trendlinie
        z = np.polyfit(sample[col], sample["residual_load"], 1)
        p = np.poly1d(z)
        x_sorted = np.sort(sample[col])
        ax.plot(x_sorted, p(x_sorted), "r-", linewidth=2, alpha=0.8)
        
        corr_val = sample[col].corr(sample["residual_load"])
        ax.set_title(f"r = {corr_val:.3f}")
        ax.grid(True, alpha=0.2)

plt.suptitle("Wettervariablen vs. Residuallast (5.000 zufaellige Stunden)", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("output/eda/06_scatter_wetter.png", bbox_inches="tight")
plt.show()

**Erkenntnisse:**
- **Windgeschwindigkeit** (100m, wind-gewichtet): Staerkste negative Korrelation — mehr Wind = deutlich weniger Residuallast
- **Globalstrahlung** (solar-gewichtet): Klar negativ — bei viel Sonnenschein produzieren Solaranlagen viel, Residuallast sinkt
- **Temperatur**: Leicht negativ — waermere Tage haben weniger Heizbedarf und gleichzeitig oft mehr Solar

## 7. Vergleich der drei Wetter-Aggregationen

Wir haben das Wetter nach drei Schemata gewichtet, weil Deutschland kein einheitliches Wetter hat:
- Windraeder stehen hauptsaechlich im **Norden** (Hamburg, Berlin, Leipzig)
- Solaranlagen stehen hauptsaechlich im **Sueden** (Bayern, Baden-Wuerttemberg)
- Der Stromverbrauch konzentriert sich im **Westen** (NRW)

Frage: Macht die unterschiedliche Gewichtung einen Unterschied?

In [ ]:
# Vergleich der drei Gewichtungen fuer Windgeschwindigkeit und Globalstrahlung
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Windgeschwindigkeit 100m
wind_cols = ["weather_load_wind_speed_100m", "weather_wind_wind_speed_100m", "weather_solar_wind_speed_100m"]
wind_labels = ["Last-gewichtet", "Wind-gewichtet", "Solar-gewichtet"]
wind_corrs = [combined[c].corr(combined["residual_load"]) for c in wind_cols]

bars = axes[0].bar(wind_labels, wind_corrs, color=["steelblue", "teal", "orange"], edgecolor="black")
axes[0].set_ylabel("Korrelation mit Residuallast")
axes[0].set_title("Windgeschwindigkeit (100m)\nWelche Gewichtung korreliert am staerksten?")
for bar, corr in zip(bars, wind_corrs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() - 0.02,
                 f"{corr:.3f}", ha="center", va="top", fontsize=12, fontweight="bold")
axes[0].grid(True, alpha=0.3, axis="y")

# Globalstrahlung
solar_cols = ["weather_load_shortwave_radiation", "weather_wind_shortwave_radiation", "weather_solar_shortwave_radiation"]
solar_corrs = [combined[c].corr(combined["residual_load"]) for c in solar_cols]

bars = axes[1].bar(wind_labels, solar_corrs, color=["steelblue", "teal", "orange"], edgecolor="black")
axes[1].set_ylabel("Korrelation mit Residuallast")
axes[1].set_title("Globalstrahlung\nWelche Gewichtung korreliert am staerksten?")
for bar, corr in zip(bars, solar_corrs):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() - 0.02,
                 f"{corr:.3f}", ha="center", va="top", fontsize=12, fontweight="bold")
axes[1].grid(True, alpha=0.3, axis="y")

plt.suptitle("Vergleich der drei Wetter-Aggregationen", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("output/eda/10_aggregation_vergleich.png", bbox_inches="tight")
plt.show()

print("Ergebnis:")
print(f"  Windgeschwindigkeit: Wind-gewichtet ({wind_corrs[1]:.3f}) > Last-gewichtet ({wind_corrs[0]:.3f}) > Solar-gewichtet ({wind_corrs[2]:.3f})")
print(f"  Globalstrahlung: Solar-gewichtet ({solar_corrs[2]:.3f}) > Last-gewichtet ({solar_corrs[0]:.3f}) > Wind-gewichtet ({solar_corrs[1]:.3f})")
print(f"\n=> Die kapazitaets-gewichteten Wetterdaten korrelieren staerker mit der Residuallast!")
print(f"   Das bestaetigt unseren Ansatz, drei verschiedene Gewichtungen zu verwenden.")

## 8. Stationaritaetstest

**Was bedeutet Stationaritaet?**

Eine Zeitreihe ist **stationaer**, wenn ihr Mittelwert und ihre Streuung sich ueber die Zeit nicht aendern. Das ist wichtig, weil viele Prognosemodelle stationaere Daten erwarten.

Der **ADF-Test** (Augmented Dickey-Fuller) prueft das:
- p-Wert < 0.05: Die Zeitreihe IST stationaer (gut!)
- p-Wert >= 0.05: Die Zeitreihe ist NICHT stationaer (muesste differenziert werden)

In [ ]:
from statsmodels.tsa.stattools import adfuller

adf_results = []
for col in ["residual_load", "total_load", "solar", "wind_total"]:
    data = smard[col].dropna()
    result = adfuller(data, maxlag=168)
    adf_results.append({
        "Variable": col,
        "ADF-Statistik": round(result[0], 2),
        "p-Wert": f"{result[1]:.2e}",
        "Stationaer?": "Ja" if result[1] < 0.05 else "Nein",
    })

adf_df = pd.DataFrame(adf_results)
print("=== Augmented Dickey-Fuller Test ===")
print("(p < 0.05 = stationaer)\n")
adf_df

**Ergebnis:** Alle Zeitreihen sind stationaer. Das bedeutet, wir koennen sie direkt fuer unsere Modelle verwenden, ohne sie vorher zu differenzieren.

## 9. Autokorrelation

**Was ist Autokorrelation?**

Autokorrelation misst, wie stark der aktuelle Wert mit vergangenen Werten zusammenhaengt:
- **Lag 1**: Wie aehnlich ist die Residuallast JETZT dem Wert vor 1 Stunde?
- **Lag 24**: ...dem Wert von GESTERN zur gleichen Uhrzeit?
- **Lag 168**: ...dem Wert von LETZTER WOCHE zur gleichen Uhrzeit?

Das hilft uns zu entscheiden, wie weit das Modell in die Vergangenheit schauen soll (`input_chunk_length`).

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

residual_data = smard["residual_load"].dropna()

# ACF (Autokorrelation)
plot_acf(residual_data, lags=336, ax=axes[0], alpha=0.05)
axes[0].set_title("Autokorrelation (ACF) — Residuallast")
axes[0].set_xlabel("Lag (Stunden)")
axes[0].axvline(x=24, color="red", linestyle="--", alpha=0.7, label="24h (1 Tag)")
axes[0].axvline(x=168, color="green", linestyle="--", alpha=0.7, label="168h (1 Woche)")
axes[0].legend(fontsize=10)

# PACF (Partielle Autokorrelation)
plot_pacf(residual_data, lags=72, ax=axes[1], alpha=0.05, method="ywm")
axes[1].set_title("Partielle Autokorrelation (PACF) — Residuallast")
axes[1].set_xlabel("Lag (Stunden)")

plt.tight_layout()
plt.savefig("output/eda/07_autokorrelation.png", bbox_inches="tight")
plt.show()

print("Autokorrelation bei wichtigen Lags:")
print(f"  Lag 1  (1 Stunde):  {residual_data.autocorr(lag=1):.3f}")
print(f"  Lag 24 (1 Tag):     {residual_data.autocorr(lag=24):.3f}")
print(f"  Lag 168 (1 Woche):  {residual_data.autocorr(lag=168):.3f}")
print(f"\n=> Starke Autokorrelation bei Lag 24 und 168 bestaetigt:")
print(f"   input_chunk_length = 168 (1 Woche) ist eine gute Wahl.")

## 10. Beispielwochen: Winter vs. Sommer

Zum Schluss ein Zoom auf je eine typische Winter- und Sommerwoche, um die Unterschiede zu verdeutlichen.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=False)

# Winterwoche (Januar 2023)
winter = smard.loc["2023-01-09":"2023-01-15"]
axes[0].plot(winter.index, winter["total_load"], label="Gesamtlast", color="gray", linewidth=1.5)
axes[0].plot(winter.index, winter["residual_load"], label="Residuallast", color="tab:blue", linewidth=2)
axes[0].fill_between(winter.index, winter["residual_load"], winter["total_load"], alpha=0.2, color="green", label="Wind+Solar")
axes[0].set_ylabel("MW")
axes[0].set_title("Winterwoche (9.-15. Januar 2023)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%a %H:%M"))

# Sommerwoche (Juli 2023)
summer = smard.loc["2023-07-10":"2023-07-16"]
axes[1].plot(summer.index, summer["total_load"], label="Gesamtlast", color="gray", linewidth=1.5)
axes[1].plot(summer.index, summer["residual_load"], label="Residuallast", color="tab:blue", linewidth=2)
axes[1].fill_between(summer.index, summer["residual_load"], summer["total_load"], alpha=0.2, color="green", label="Wind+Solar")
axes[1].axhline(y=0, color="red", linestyle="--", linewidth=0.8, alpha=0.5)
axes[1].set_ylabel("MW")
axes[1].set_title("Sommerwoche (10.-16. Juli 2023)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%a %H:%M"))

plt.tight_layout()
plt.savefig("output/eda/08_beispielwochen.png", bbox_inches="tight")
plt.show()

## Zusammenfassung

### Zentrale Erkenntnisse fuer die Modellierung:

1. **Starke Muster**: Die Residuallast folgt klaren Tages-, Wochen- und Jahreszeitenmustern. Das ist gut — Muster koennen Modelle lernen!

2. **Wind ist der wichtigste Treiber**: Die Windgeschwindigkeit hat die staerkste Korrelation mit der Residuallast (-0.45). Ein windiger Tag kann die Residuallast um 20.000+ MW druecken.

3. **Drei Gewichtungen lohnen sich**: Die kapazitaets-gewichteten Wetterdaten (Wind-gewichtet fuer Windgeschwindigkeit, Solar-gewichtet fuer Strahlung) korrelieren staerker mit der Residuallast als ein einheitlicher Durchschnitt.

4. **Lookback = 1 Woche**: Die starke Autokorrelation bei Lag 168 bestaetigt, dass das Modell mindestens 1 Woche in die Vergangenheit schauen sollte.

5. **Stationaer**: Keine Differenzierung noetig — die Daten koennen direkt verwendet werden.

6. **Negative Residuallast**: In ca. 0.7% der Stunden ist die Residuallast negativ. Die Modelle muessen also auch negative Werte vorhersagen koennen.

---

**Weiter:** Walkthrough-Notebook (`00_walkthrough.ipynb`) fuer die Modellierung.